# Project: Student Performance Analysis

**Objective:** Analyze student academic and lifestyle data to identify factors associated with exam performance, explore the relationship between study hours and scores, and rank students by relative exam performance.

**Business Questions:**

1. Do students who study more than 10 hours per week and participate in extracurricular activities achieve higher exam scores?
2. Is there a study-hour range associated with higher average exam performance?
3. How can students be ranked by exam performance while keeping their actual exam scores private?

**Table Used:**

* `student_performance`

  * `attendance` : Percentage of classes attended
  * `extracurricular_activities` : Whether the student participates in extracurricular activities (`Yes`, `No`)
  * `sleep_hours` : Average number of hours of sleep per night
  * `tutoring_sessions` : Number of tutoring sessions attended per month
  * `teacher_quality` : Teacher quality rating (`Low`, `Medium`, `High`)
  * `exam_score` : Final exam score

**Methodology:**

1. Perform an initial exploration of the dataset by reviewing student records ordered by `hours_studied`.
2. Filter students who study more than 10 hours per week and participate in extracurricular activities.
3. Calculate the average exam score for each `hours_studied` value to examine how study time relates to performance among these students.
4. Categorize students into four study-hour groups:

   * `1-5 hours`
   * `6-10 hours`
   * `11-15 hours`
   * `16+ hours`
5. Calculate the average exam score for each study-hour range to identify which study range has the highest average performance.
6. Use the `DENSE_RANK()` window function to rank students according to `exam_score`, with students receiving the same score assigned the same rank and no rank numbers skipped.
7. Sort students by rank so that the highest-performing students appear first, and return the top 30 ranked students without exposing their actual exam scores.

**Output Columns:**

**Analysis 1 — Study Hours & Extracurricular Activities**

* `hours_studied` : Number of hours studied per week
* `avg_exam_score` : Average exam score for students who study more than 10 hours and participate in extracurricular activities

**Analysis 2 — Study-Hour Ranges**

* `hours_studied_range` : Categorized study-hour range
* `avg_exam_score` : Average exam score for each study-hour range

**Analysis 3 — Student Ranking**

* `attendance` : Percentage of classes attended
* `hours_studied` : Number of hours studied per week
* `sleep_hours` : Average hours of sleep per night
* `tutoring_sessions` : Number of tutoring sessions attended per month
* `exam_rank` : Relative student rank based on exam score

**SQL Techniques Used:**

* `SELECT`
* `WHERE` filtering
* `GROUP BY`
* `ORDER BY`
* `LIMIT`
* Aggregate functions with `AVG()`
* Conditional logic with `CASE`
* Window functions
* `DENSE_RANK()`
* `OVER()`
* Ranking with `ORDER BY`
* Data categorization into study-hour ranges
* Filtering and aggregation for performance analysis


In [1]:
-- View the first 10 records, ordered by hours_studied
SELECT *
FROM student_performance
ORDER BY hours_studied
LIMIT 10;

,hours_studied,attendance,extracurricular_activities,sleep_hours,tutoring_sessions,exam_score
0,1,69,Yes,6,1,61
1,1,88,Yes,4,3,92
2,1,81,Yes,8,1,60
3,2,99,Yes,9,0,62
4,2,67,No,6,1,58
5,2,98,No,4,1,65
6,2,96,Yes,9,3,65
7,2,84,No,8,3,62
8,2,98,Yes,7,2,66
9,3,62,No,6,1,55


In [2]:
-- 1. Analyze how studying more than 10 hours per week while also  
-- participating in extracurricular activities affects exam scores.
SELECT
	hours_studied,
	-- Calculate the average exam score
	AVG(exam_score) AS avg_exam_score
FROM student_performance
-- Filter to only include study hours greater than 10
WHERE hours_studied > 10 
	-- and those who participate in extracurricular activities
	AND extracurricular_activities = 'Yes'
-- Group the result by hours_studied
GROUP BY hours_studied
-- Sort the result by hours_studied
-- from highest to lowest
ORDER BY hours_studied DESC;

,hours_studied,avg_exam_score
0,43,78.000000
1,39,75.000000
2,38,73.500000
3,37,73.000000
4,36,70.428571
5,35,72.312500
6,34,71.187500
7,33,70.333333
8,32,71.325000
9,31,70.553191


In [3]:
-- 2. Explore how different study hour ranges
-- affect exam scores.
SELECT
	CASE
		-- Group study hours into different ranges
		WHEN hours_studied BETWEEN 1 AND 5 THEN '1-5 hours'
		WHEN hours_studied BETWEEN 6 AND 10 THEN '6-10 hours'
		WHEN hours_studied BETWEEN 11 AND 15 THEN '11-15 hours'
		ELSE '16+ hours'
	END AS hours_studied_range,
	-- Calculate the average of exam scores
	AVG(exam_score) AS avg_exam_score
FROM student_performance
-- Group the result by the newly created study ranges
GROUP BY hours_studied_range
-- Sort the result by the average exam score
-- from highest to lowest 
ORDER BY avg_exam_score DESC

,hours_studied_range,avg_exam_score
0,16+ hours,67.923363
1,11-15 hours,65.204386
2,6-10 hours,64.225490
3,1-5 hours,62.627119


In [4]:
-- 3. Rank students based on exam scores
SELECT
	attendance,
	hours_studied,
	sleep_hours,
	tutoring_sessions,
	-- Calculate rank using DENSE_RANK() so tied scores are shared
	-- and rank numbers aren't skipped
	DENSE_RANK() OVER(ORDER BY exam_score DESC) AS exam_rank
FROM student_performance
-- Sort the result by the newly created exam_rank
-- from lowest to highest
ORDER BY exam_rank ASC
-- Return the first 30 rows after sorting by exam_rank
LIMIT 30

,attendance,hours_studied,sleep_hours,tutoring_sessions,exam_rank
0,98,27,6,5,1
1,89,18,4,3,2
2,90,14,8,4,3
3,83,23,4,1,3
4,96,28,4,1,4
5,90,28,9,0,4
6,83,16,8,2,4
7,83,15,7,2,5
8,74,21,6,1,5
9,99,25,7,0,5
